## Arquitectura del entorno de desarrollo

Antes de comenzar con la instalación, es importante comprender cómo se relacionan los componentes que vamos a configurar. El siguiente diagrama muestra la **arquitectura del entorno** que construiremos a lo largo de esta lección:

```mermaid
flowchart TB
    subgraph Sistema["Sistema operativo"]
        UV["uv (gestor de paquetes)"]
        Python["Python 3.14"]
        Ollama["Ollama Server"]
    end
    
    subgraph Proyecto["Proyecto Python"]
        VENV[".venv (entorno virtual)"]
        LC["langchain"]
        LCO["langchain-ollama"]
    end
    
    UV --> Python
    UV --> VENV
    VENV --> LC
    VENV --> LCO
    LCO <--> Ollama
    
    style Sistema fill:#5555,color:#fff
    style Proyecto fill:#3b82f6,color:#fff
    style Ollama fill:#10b981,color:#fff
```

El flujo de trabajo que estableceremos permite que **LangChain** se comunique con **Ollama** para ejecutar modelos de lenguaje de forma local, sin necesidad de conexión a servicios en la nube. Esta configuración es ideal para desarrollo, experimentación y aplicaciones que requieren privacidad de datos.

> Un entorno bien configurado es la base de cualquier proyecto de IA Generativa. Invertir tiempo en esta fase inicial ahorra horas de depuración posteriormente.

## Instalación de uv

**uv** es un gestor de paquetes y entornos virtuales para Python desarrollado por Astral. Escrito en Rust, destaca por su velocidad y por simplificar la gestión de versiones de Python y dependencias en un único comando.

### Instalación en Windows

Abre **PowerShell** como administrador y ejecuta los siguientes comandos:

In [ ]:
%%powershell
Set-ExecutionPolicy -Scope Process Bypass
irm https://astral.sh/uv/install.ps1 | iex

El primer comando permite la ejecución de scripts en la sesión actual. El segundo descarga e instala uv automáticamente.

### Instalación en macOS y Linux

Abre una terminal y ejecuta:

In [ ]:
curl -LsSf https://astral.sh/uv/install.sh | sh

### Verificación de la instalación

Una vez completada la instalación, verifica que uv está disponible:

In [ ]:
uv --version

Deberías ver un mensaje con la versión instalada, algo similar a `uv 0.8.x`.

> Si el comando no se reconoce, cierra y vuelve a abrir la terminal para que los cambios en el PATH surtan efecto.

## Instalación de Python con uv

Una de las ventajas de uv es que puede **gestionar múltiples versiones de Python** sin conflictos. Esto elimina la necesidad de instaladores manuales y permite cambiar entre versiones según el proyecto.

Para instalar Python 3.14, ejecuta:

In [ ]:
uv python install 3.14

Este comando descarga e instala Python 3.14 en un directorio gestionado por uv. Puedes verificar las versiones disponibles con:

In [ ]:
uv python list

El siguiente diagrama ilustra cómo uv gestiona las versiones de Python de forma aislada:

```mermaid
flowchart LR
    UV["uv"] --> P312["Python 3.12"]
    UV --> P313["Python 3.13"]
    UV --> P314["Python 3.14"]
    
    P313 --> Proj["Tu proyecto"]
    
    style UV fill:#f59e0b,color:#000
    style P313 fill:#10b981,color:#fff
    style Proj fill:#3b82f6,color:#fff
```

> uv almacena las versiones de Python en `~/.local/share/uv/python` en Linux/macOS y en `%LOCALAPPDATA%\uv\python` en Windows, manteniendo el sistema limpio.

## Creación del proyecto y entorno virtual

Con uv instalado, vamos a crear un **proyecto nuevo** con su entorno virtual aislado. Los entornos virtuales permiten que cada proyecto tenga sus propias dependencias sin interferir con otros proyectos.

### Crear el directorio del proyecto

In [ ]:
mkdir genai-curso
cd genai-curso

### Inicializar el proyecto con uv

uv puede inicializar un proyecto completo con la estructura estándar de Python:

In [ ]:
uv init --python 3.14

Este comando crea:

* Un archivo `pyproject.toml` con la configuración del proyecto
* Un archivo `main.py` de ejemplo
* Un archivo `.python-version` que indica la versión de Python a usar

### Crear y activar el entorno virtual

Crea el entorno virtual especificando la versión de Python:

In [ ]:
uv venv --python 3.14

Esto genera una carpeta `.venv` con una instalación aislada de Python. Para **activar** el entorno:

**En Windows (PowerShell):**

In [ ]:
%%powershell
.\.venv\Scripts\Activate.ps1

**En Windows (CMD):**

```cmd
.\.venv\Scripts\activate.bat
```

**En macOS y Linux:**

In [ ]:
source .venv/bin/activate

Al activar el entorno, el prompt de la terminal mostrará el prefijo `(.venv)`, indicando que estás trabajando dentro del entorno virtual.

> Siempre activa el entorno virtual antes de instalar paquetes o ejecutar código. De lo contrario, las dependencias se instalarán globalmente.

## Instalación de LangChain y langchain-ollama

**LangChain** es un framework que simplifica la construcción de aplicaciones con modelos de lenguaje. El paquete **langchain-ollama** proporciona la integración específica para comunicarse con el servidor Ollama.

Con el entorno virtual activado, instala ambas bibliotecas:

In [ ]:
uv pip install langchain langchain-ollama

uv instalará las dependencias de forma optimizada. Puedes verificar la instalación listando los paquetes:

In [ ]:
uv pip list

Deberías ver `langchain` y `langchain-ollama` junto con sus dependencias. El archivo `pyproject.toml` se actualizará automáticamente si usas `uv add` en lugar de `uv pip install`:

In [ ]:
uv add langchain langchain-ollama

La diferencia es que `uv add` registra las dependencias en `pyproject.toml`, facilitando la reproducibilidad del proyecto.

## Instalación y ejecución de Ollama

**Ollama** es una plataforma de código abierto que permite ejecutar modelos de lenguaje de gran tamaño de forma local. Actúa como un servidor que expone una API REST, con la que LangChain se comunica para enviar prompts y recibir respuestas.

### Instalación en Windows

Descarga el instalador desde la página oficial:

* [https://ollama.com/download](https://ollama.com/download)

Ejecuta el instalador y sigue las instrucciones. Una vez completado, Ollama se ejecutará como un servicio en segundo plano.

### Instalación en macOS

Descarga la aplicación desde [https://ollama.com/download](https://ollama.com/download) o usa Homebrew:

In [ ]:
brew install ollama

### Instalación en Linux

Ejecuta el script de instalación oficial:

In [ ]:
curl -fsSL https://ollama.com/install.sh | sh

### Verificación de la instalación

Comprueba que Ollama está instalado correctamente:

In [ ]:
ollama --version

### Iniciar el servidor Ollama

En **Windows**, Ollama se inicia automáticamente como servicio tras la instalación. En **macOS y Linux**, inicia el servidor manualmente si no está corriendo:

In [ ]:
ollama serve

Este comando arranca el servidor en `http://localhost:11434`. Puedes verificar que está funcionando accediendo a esa URL desde un navegador o con curl:

In [ ]:
curl http://localhost:11434

Si recibes una respuesta como `Ollama is running`, el servidor está listo.

```mermaid
sequenceDiagram
    participant App as Tu aplicación
    participant LC as LangChain
    participant OL as Ollama Server
    participant Model as Modelo LLM
    
    App->>LC: Envía prompt
    LC->>OL: POST /api/generate
    OL->>Model: Procesa prompt
    Model-->>OL: Genera respuesta
    OL-->>LC: Devuelve respuesta
    LC-->>App: Retorna resultado
```

> Ollama escucha por defecto en el puerto 11434. Asegúrate de que ningún otro servicio esté usando ese puerto antes de iniciar el servidor.

## Verificación del entorno completo

Para confirmar que todo está configurado correctamente, crea un archivo `verificar_entorno.py` con el siguiente contenido:

In [ ]:
%%python
# verificar_entorno.py
import sys

print(f"Python version: {sys.version}")

try:
    import langchain
    print(f"LangChain instalado: {langchain.__version__}")
except ImportError:
    print("LangChain NO instalado")

try:
    import langchain_ollama
    print("langchain-ollama instalado correctamente")
except ImportError:
    print("langchain-ollama NO instalado")

# Verificar conexión con Ollama
import urllib.request
try:
    response = urllib.request.urlopen("http://localhost:11434")
    print("Ollama server: ACTIVO")
except Exception as e:
    print(f"Ollama server: NO DISPONIBLE ({e})")

Ejecuta el script con:

In [ ]:
python verificar_entorno.py

Si todo está correcto, verás mensajes confirmando la versión de Python, la instalación de LangChain y langchain-ollama, y que el servidor Ollama está activo.
